# PSL Win Predictor — Exploratory Data Analysis

This notebook covers the actual data science work behind the project: examining the raw PSL match data before any modeling happens, checking what patterns actually exist, and confirming the engineered features hold up before trusting them in a model.

Run this after downloading `psl_matches.csv` into `../data/`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## 1. Load and inspect the raw data

In [3]:
df = pd.read_csv('../data/PSL_Match_Results.csv')
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/PSL_Match_Results.csv'

In [4]:
# Missing values -- always check this before trusting anything downstream
df.isnull().sum()

NameError: name 'df' is not defined

## 2. Does winning the toss actually predict winning the match?

In [ ]:
df['toss_led_to_win'] = df['toss_winner'] == df['winner']
toss_win_rate = df['toss_led_to_win'].mean()
print(f"Toss winner also won the match: {toss_win_rate:.1%} of the time")

sns.countplot(x='toss_led_to_win', data=df)
plt.title('Did the toss winner also win the match?')
plt.xlabel('Toss winner = Match winner')
plt.show()

If this comes out close to 50%, the toss doesn't meaningfully predict outcomes on its own -- which is expected and fine. It's one signal among several, not a standalone predictor. Don't overstate this if the number isn't dramatic.

## 3. Team win rates overall

In [ ]:
wins = df['winner'].value_counts()
plt.figure(figsize=(9, 5))
sns.barplot(x=wins.values, y=wins.index, orient='h')
plt.title('Total match wins by team')
plt.xlabel('Wins')
plt.show()

## 4. Venue win rates -- is there a real home-ground advantage?

In [ ]:
venue_matches = df['venue'].value_counts()
plt.figure(figsize=(9, 5))
sns.barplot(x=venue_matches.values, y=venue_matches.index, orient='h')
plt.title('Matches played per venue')
plt.xlabel('Matches')
plt.show()

## 5. Engineered features — sanity check

These are the same features `preprocess.py` builds for the model: recent form, head-to-head win rate, and venue win rate. Worth visualizing here to confirm they behave sensibly before trusting them in training.

In [ ]:
from preprocess import build_features

feat_df, feature_cols = build_features('../data/psl_matches.csv')
feat_df[feature_cols + ['target']].describe()

In [ ]:
# Correlation between engineered features and the target
corr = feat_df[feature_cols + ['target']].corr()['target'].drop('target')
plt.figure(figsize=(7, 4))
corr.sort_values().plot(kind='barh')
plt.title('Feature correlation with match outcome (team1 win)')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

This is the honest check: if none of the engineered features correlate meaningfully with the outcome, the model built from them won't perform well either, and that's worth knowing *before* training rather than being surprised by a low accuracy number later.

## 6. Train the model

Actual training happens in `src/train_model.py` (kept separate so the app can import a clean script). This cell just re-runs it here for a self-contained notebook view of the result.

In [ ]:
%cd ../src
%run train_model.py
%cd ../notebooks

## Notes

- Features are built only from data available *before* each match (no leakage).
- Model accuracy reflects what's realistically achievable from match-result history alone -- it does not account for player injuries, current squad changes, or weather, since none of that is present in this dataset.